In [1]:
# Installation des bibliothèques nécessaires
!pip install -q ultralytics gradio opencv-python-headless Pillow

# Vérification GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device disponible : {device}')
if device == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.7 MB/s eta 0:00:00
✅ Device disponible : cuda
   GPU : Tesla T4
   VRAM : 15.6 Go


In [3]:
from google.colab import drive
import os  # ✅ AJOUT IMPORTANT

drive.mount('/content/drive')

# Adaptez ce chemin selon où vous avez mis le fichier dans Drive :
MODEL_PATH = '/content/drive/MyDrive/gorilla_detection/best_model.pt'

# Vérification
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f'✅ Modèle trouvé : {MODEL_PATH}  ({size_mb:.1f} Mo)')
else:
    print(f'❌ Modèle introuvable : {MODEL_PATH}')
    print("→ Vérifiez le chemin ou utilisez l'Option A (upload manuel)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Modèle trouvé : /content/drive/MyDrive/gorilla_detection/best_model.pt  (5.5 Mo)


In [4]:
import time
import torch
import numpy as np
import cv2
from PIL import Image
from ultralytics import YOLO

# 1. Configuration du Système Expert (selon votre code)
class GorillaExpertSystem:
    def __init__(self, MODEL_PATH):
        self.model = YOLO(MODEL_PATH)
        self.gamma_threshold = 0.50 # Votre seuil de décision final Γ

    def calculer_descripteurs(self, crop):
        """Intègre vos formules : C1 (Dos Argenté), C2 (Pelage), C3 (Facial)"""
        if crop.size == 0: return 0, 0, 0

        # C2 - FTDD : Densité du Pelage (Var. Laplacien)
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        c2_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        c2 = 1.0 if c2_score > 500 else 0.0

        # C1 & C3 (Simulés ici pour le benchmark de vitesse)
        # Dans votre code, ces calculs utilisent la luminance et les contours
        c1 = 0.75 # Exemple DCID
        c3 = 0.60 # Exemple FSGD

        return c1, c2, c3

    def fusion_decisionnelle(self, score_ia, c1, c2, c3):
        """Votre formule : Decision = 0.60·ScoreIA + 0.40·S_E"""
        s_e = 0.40*c1 + 0.20*c2 + 0.40*c3
        score_final = 0.60*score_ia + 0.40*s_e
        return score_final >= self.gamma_threshold

    def pipeline_complet(self, frame):
        # A. Inférence YOLO (Détection)
        results = self.model(frame, verbose=False)[0]

        for box in results.boxes:
            score_ia = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = frame[y1:y2, x1:x2]

            # B. Validation par le Système Expert
            c1, c2, c3 = self.calculer_descripteurs(crop)
            valide = self.fusion_decisionnelle(score_ia, c1, c2, c3)

        return results

# 2. Exécution du Benchmark sur Colab
def run_scientific_benchmark(model_path, iterations=100):
    detector = GorillaExpertSystem(MODEL_PATH)
    dummy_frame = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

    # Warm-up
    print("Échauffement du GPU...")
    for _ in range(10): _ = detector.pipeline_complet(dummy_frame)

    print(f"Lancement du test sur {iterations} itérations...")
    torch.cuda.synchronize() # Synchronisation pour mesure précise sur GPU
    start = time.time()

    for _ in range(iterations):
        _ = detector.pipeline_complet(dummy_frame)

    torch.cuda.synchronize()
    end = time.time()

    total_time = end - start
    fps = iterations / total_time
    print(f"\n✅ RÉSULTAT : {fps:.2f} FPS")
    print(f"⏱️ Temps par image : {(total_time/iterations)*1000:.2f} ms")

# Utilisation (Assurez-vous d'avoir chargé votre modèle .pt)
# run_scientific_benchmark("votre_modele.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
import time
import torch
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

class GorillaExpertBenchmark:
    def __init__(self, MODEL_PATH):
        # Chargement du modèle YOLOv11n
        self.model = YOLO(MODEL_PATH)
        self.gamma_threshold = 0.50 # Votre seuil Γ définit dans l'article

    def calculer_descripteurs(self, crop):
        """
        Implémentation de vos fonctions expertes :
        C1 (DCID), C2 (FTDD), C3 (FSGD)
        """
        if crop.size == 0: return 0.5, 0, 0.5

        # Conversion pour les calculs morphologiques
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        # C2 - FTDD : Densité du Pelage (Variance du Laplacien)
        # C'est l'opération la plus coûteuse en calcul
        c2_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        c2 = 1.0 if c2_score > 500 else 0.0

        # C1 - DCID : Simulation du Dos Argenté (basé sur la luminance L̄)
        l_mean = np.mean(gray) / 255.0
        c1 = min(1.0, 1.2 * l_mean) # Simplification de votre formule DCID pour le test

        # C3 - FSGD : Morphologie Faciale (basé sur la densité de contours)
        edges = cv2.Canny(gray, 100, 200)
        c3 = np.count_nonzero(edges) / (gray.shape[0] * gray.shape[1])
        c3 = min(1.0, c3 * 5) # Normalisation FSGD

        return c1, c2, c3

    def fusion_decisionnelle(self, score_ia, c1, c2, c3):
        """Votre formule : 0.60·ScoreIA + 0.40·S_E"""
        s_e = (0.40 * c1) + (0.20 * c2) + (0.40 * c3)
        score_final = (0.60 * score_ia) + (0.40 * s_e)
        return score_final >= self.gamma_threshold

    def inference_hybride(self, frame):
        # 1. Détection YOLO
        results = self.model(frame, verbose=False)[0]

        # 2. Validation par le Système Expert pour chaque détection
        for box in results.boxes:
            score_ia = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Découpe (Crop) de la zone détectée
            crop = frame[y1:y2, x1:x2]

            if crop.size > 0:
                c1, c2, c3 = self.calculer_descripteurs(crop)
                _ = self.fusion_decisionnelle(score_ia, c1, c2, c3)

        return len(results.boxes)

# --- CONFIGURATION DU TEST ---
# Remplacez 'best.pt' par le nom de votre fichier téléchargé dans Colab
MODEL_PATH =  '/content/drive/MyDrive/gorilla_detection/best_model.pt'

if Path(MODEL_PATH).exists():
    expert_system = GorillaExpertBenchmark(MODEL_PATH)

    # Création d'une image de test (640x640)
    test_frame = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

    print("🚀 Échauffement du GPU (Warm-up)...")
    for _ in range(20): _ = expert_system.inference_hybride(test_frame)

    print("⏱️ Lancement du Benchmark (100 itérations)...")
    nb_tests = 100
    torch.cuda.synchronize() # Synchronisation précise pour GPU
    start_time = time.time()

    for _ in range(nb_tests):
        _ = expert_system.inference_hybride(test_frame)

    torch.cuda.synchronize()
    end_time = time.time()

    # Calcul des résultats
    total_duration = end_time - start_time
    fps_final = nb_tests / total_duration
    ms_per_image = (total_duration / nb_tests) * 1000

    print("\n" + "="*30)
    print(f"📊 RÉSULTATS DU SYSTÈME HYBRIDE")
    print("="*30)
    print(f"Vitesse : {fps_final:.2f} FPS")
    print(f"Latence : {ms_per_image:.2f} ms / image")
    print("="*30)

    if fps_final >= 107:
        print("✅ PERFORMANCE CONFIRMÉE : Vous atteignez l'objectif de votre article !")
    else:
        print(f"ℹ️ Résultat actuel : {fps_final:.2f} FPS. Note : La vitesse dépend du GPU alloué par Colab.")
else:
    print(f"❌ Erreur : Le fichier {MODEL_PATH} est introuvable. Veuillez l'importer dans Colab.")

🚀 Échauffement du GPU (Warm-up)...
⏱️ Lancement du Benchmark (100 itérations)...

📊 RÉSULTATS DU SYSTÈME HYBRIDE
Vitesse : 101.33 FPS
Latence : 9.87 ms / image
ℹ️ Résultat actuel : 101.33 FPS. Note : La vitesse dépend du GPU alloué par Colab.
